### 1. Higienização e Tradução: Informações dos Filmes
Nesta primeira etapa da camada Silver, realizei a padronização estrutural dos metadados dos filmes:
* **Deduplicação:** Implementação de uma *Window Function* particionada pelo ID do filme para manter apenas o registro com a data de ingestão mais recente, garantindo unicidade.
* **Tratamento Multi-Formato:** Utilização de `coalesce` aliado a `try_to_date` para testar múltiplos padrões de data de forma segura, sem quebrar o pipeline.
* **Normalização de Strings:** Aplicação de expressões regulares (Regex) para remover ruídos da coluna de status, seguida da tradução parametrizada via `when().otherwise()` para o português.

In [0]:
from pyspark.sql.functions import col, when, trim, lower, regexp_replace, coalesce, year, row_number, expr
from pyspark.sql.window import Window

# Topo da Célula 1
spark.sql("USE CATALOG cinedata_analytics")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# 2. Ler os dados da camada Bronze
df_info = spark.read.table("bronze.tb_movies_info")

# 3. Deduplicação: Unidade por filme baseada na data de ingestão mais recente
window_spec = Window.partitionBy("id").orderBy(col("ingestion_datetime").desc())
df_dedup = df_info.withColumn("rn", row_number().over(window_spec)) \
                  .filter(col("rn") == 1) \
                  .drop("rn", "ingestion_datetime")

# 4. Tratamento de Data Multi-Formato seguro (tolerância a falhas via try_to_date)
data_formatada = coalesce(
    expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    expr("try_to_date(release_date, 'dd/MM/yyyy')"),
    expr("try_to_date(release_date, 'MM/dd/yyyy')")
)

# 5. Limpeza e Tradução do Status
status_limpo = lower(trim(regexp_replace(col("status"), r"[^a-zA-Z\s]", "")))

traducao_status = when(status_limpo == "released", "Lançado") \
                 .when(status_limpo == "post production", "Pós-Produção") \
                 .when(status_limpo == "in production", "Em Produção") \
                 .when(status_limpo == "planned", "Planejado") \
                 .when(status_limpo == "rumored", "Rumores") \
                 .when(status_limpo == "canceled", "Cancelado") \
                 .otherwise("Não Informado")

# 6. Seleção, Mapeamento de Colunas e Coluna Derivada
# 6. Seleção, Mapeamento de Colunas e Coluna Derivada
df_silver_info = df_dedup.select(
    col("id").alias("id_filme"),
    col("title").alias("titulo"),
    col("original_title").alias("titulo_original"),
    data_formatada.alias("data_lancamento"),
    year(data_formatada).alias("ano_lancamento"), 
    expr("try_cast(runtime as int)").alias("duracao_minutos"), # <-- Correção aplicada aqui
    col("original_language").alias("idioma_original"),
    traducao_status.alias("status_filme"),
    col("overview").alias("sinopse"),
    col("tagline").alias("frase_divulgacao")
)

# 7. Gravar na camada Silver
df_silver_info.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")
print("Tabela silver.tb_info_filmes processada e gravada com sucesso!")

Tabela silver.tb_info_filmes processada e gravada com sucesso!


### 2. Tratamento Financeiro e Conversão Monetária
Esta célula é responsável por processar o orçamento e a receita, garantindo precisão matemática (decimal) e adequação ao câmbio:
* **Busca de Cotação:** Extração do valor da cotação mais recente já processada na tabela Silver de PTAX.
* **Limpeza de Caracteres:** Uso de Regex (`[^0-9.]`) para expurgar símbolos de moedas e pontuações indesejadas dos dados brutos.
* **Regras de Negócio e Derivações:** Cálculo dos valores em Reais (BRL) aplicando a cotação como constante (`lit()`), além da geração das colunas de Lucro e Margem Percentual, utilizando tratamento explícito para evitar divisão por zero.
* **Garantia de Unicidade:** Aplicação de `dropDuplicates` para evitar o temido *Join Explosion* na camada Gold.

In [0]:
from pyspark.sql.functions import col, when, regexp_replace, expr, round, lit

# 1. Ler a tabela Silver de cotação (Provando que usamos o Forward Fill gerado no Item 7)
df_cotacao_silver = spark.read.table("silver.tb_cotacao_dolar")

# 2. Obter a cotação mais recente disponível na tabela tratada
cotacao_atual = df_cotacao_silver.orderBy(col("data_cotacao").desc()).select("cotacao_dolar").first()[0]

# 3. Ler e limpar a tabela financeira bruta (Higienização exigida no Item 2)
df_fin_bruta = spark.read.table("bronze.tb_movies_financials")
limpeza_regex = r"[^0-9.]"

df_fin_tratada = df_fin_bruta.withColumn("orcamento_usd", expr(f"try_cast(regexp_replace(budget, '{limpeza_regex}', '') as decimal(18,2))")) \
                             .withColumn("receita_usd", expr(f"try_cast(regexp_replace(revenue, '{limpeza_regex}', '') as decimal(18,2))")) \
                             .withColumn("orcamento_usd", when(col("orcamento_usd") > 0, col("orcamento_usd")).otherwise(None)) \
                             .withColumn("receita_usd", when(col("receita_usd") > 0, col("receita_usd")).otherwise(None))

# 4. Calcular Valores BRL e Derivados (Garantindo unicidade por filme para evitar Join Explosion)
df_silver_financeiro = df_fin_tratada.select(
    col("id").alias("id_filme"),
    col("orcamento_usd"),
    col("receita_usd"),
    round(col("orcamento_usd") * lit(cotacao_atual), 2).alias("orcamento_brl"),
    round(col("receita_usd") * lit(cotacao_atual), 2).alias("receita_brl"),
    (col("receita_usd") - col("orcamento_usd")).alias("lucro_usd"),
    (round(col("receita_usd") * lit(cotacao_atual), 2) - round(col("orcamento_usd") * lit(cotacao_atual), 2)).alias("lucro_brl"),
    when(col("receita_usd") > 0, round(((col("receita_usd") - col("orcamento_usd")) / col("receita_usd")) * 100, 2)).otherwise(None).alias("margem_lucro_pct")
).dropDuplicates(["id_filme"]) # <-- Cumpre a exigência de unicidade

df_silver_financeiro.write.format("delta").mode("overwrite").saveAsTable("silver.tb_financeiro_filmes")
print("Tabela silver.tb_financeiro_filmes processada e gravada com sucesso!")

Tabela silver.tb_financeiro_filmes processada e gravada com sucesso!


### 3. Validação de Métricas e Resiliência a Column Shift
Neste bloco, garantimos a integridade dos dados numéricos que sofreram deslocamento de colunas (*Column Shift*) na origem:
* **Conversão Segura:** Aplicação de `try_cast` para forçar a tipagem correta (Double/Int). Qualquer texto deslocado que não seja numérico é convertido silenciosamente para `NULL`, evitando falhas de execução.
* **Limites de Negócio:** Implementação de filtros lógicos (`between` e `>= 0`) para anular votos negativos e garantir que as notas do TMDB e IMDb estejam estritamente dentro da escala de 0 a 10.

In [0]:
from pyspark.sql.functions import col, when, expr, regexp_replace

# 1. Ler a tabela bruta de métricas
df_metrics_bruta = spark.read.table("bronze.tb_movies_metrics")

# 2. Limpeza da formatação numérica da Popularidade
# Substitui vírgulas por pontos e remove qualquer caractere que não seja número, ponto ou sinal negativo
df_metrics_limpa = df_metrics_bruta.withColumn(
    "pop_regex", 
    regexp_replace(regexp_replace(col("popularity"), ",", "."), r"[^0-9.-]", "")
)

# 3. Tratamento de Column Shift (Conversão Segura) e Renomeação
df_metrics_tipada = df_metrics_limpa.select(
    col("id").alias("id_filme"),
    expr("try_cast(pop_regex as double)").alias("popularidade"),
    expr("try_cast(vote_average as double)").alias("nota_media_tmdb"),
    expr("try_cast(vote_count as int)").alias("qtd_votos_tmdb"),
    expr("try_cast(averageRating as double)").alias("nota_media_imdb"),
    expr("try_cast(numVotes as int)").alias("qtd_votos_imdb")
)

# 4. Aplicação das Regras de Negócio (Limites numéricos)
df_silver_metrics = df_metrics_tipada.select(
    col("id_filme"),
    # Popularidade e Votos não podem ser negativos
    when(col("popularidade") >= 0, col("popularidade")).otherwise(None).alias("popularidade"),
    when(col("qtd_votos_tmdb") >= 0, col("qtd_votos_tmdb")).otherwise(None).alias("qtd_votos_tmdb"),
    when(col("qtd_votos_imdb") >= 0, col("qtd_votos_imdb")).otherwise(None).alias("qtd_votos_imdb"),
    
    # Notas devem estar estritamente entre 0 e 10
    when(col("nota_media_tmdb").between(0, 10), col("nota_media_tmdb")).otherwise(None).alias("nota_media_tmdb"),
    when(col("nota_media_imdb").between(0, 10), col("nota_media_imdb")).otherwise(None).alias("nota_media_imdb")
)

# 5. Gravar na camada Silver
df_silver_metrics.write.format("delta").mode("overwrite").saveAsTable("silver.tb_metricas_engajamento")
print("Tabela silver.tb_metricas_engajamento processada e gravada com sucesso!")

Tabela silver.tb_metricas_engajamento processada e gravada com sucesso!


### 4. Deduplicação e Padronização de Avaliações (Reviews)
Processamento das opiniões individuais dos usuários sobre os filmes:
* **Limpeza de Duplicatas Exatas:** Remoção de registros onde a combinação de Filme, Usuário, Nota e Comentário é 100% idêntica.
* **Validação Numérica:** Descarte de notas fora da escala de 0 a 10.
* **Preenchimento de Nulos:** Identificação de comentários vazios, nulos ou preenchidos apenas com espaços em branco, substituindo-os pelo texto padrão "Sem comentário".

In [0]:
from pyspark.sql.functions import col, when, trim, expr, to_date, last
from pyspark.sql.window import Window

# ==============================================================================
# Tabela: silver.tb_avaliacoes_usuarios
# ==============================================================================
df_reviews_bruta = spark.read.table("bronze.tb_movies_reviews")

# Renomear colunas e usar try_cast para ignorar textos na coluna de nota
df_reviews_renomeada = df_reviews_bruta.select(
    col("id").alias("id_filme"),
    col("nome").alias("nome_usuario"),
    expr("try_cast(nota as double)").alias("nota_usuario"),
    col("comentario").alias("comentario_usuario")
)

# Remover registros integralmente duplicados
df_reviews_dedup = df_reviews_renomeada.dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])

# Aplicar regras de negócio
df_silver_reviews = df_reviews_dedup.select(
    col("id_filme"),
    col("nome_usuario"),
    # Validação de nota entre 0 e 10
    when(col("nota_usuario").between(0, 10), col("nota_usuario")).otherwise(None).alias("nota_usuario"),
    # Trim remove espaços; se ficar vazio ou for nulo, recebe o texto padrão
    when((col("comentario_usuario").isNull()) | (trim(col("comentario_usuario")) == ""), "Sem comentário")
    .otherwise(trim(col("comentario_usuario"))).alias("comentario_usuario")
)

df_silver_reviews.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")
print("Tabela silver.tb_avaliacoes_usuarios processada e gravada com sucesso!")

Tabela silver.tb_avaliacoes_usuarios processada e gravada com sucesso!


### 5. Preenchimento de Série Temporal (Forward Fill): Cotação do Dólar
A API do Banco Central não fornece cotações em finais de semana ou feriados. Para garantir que o cruzamento financeiro de qualquer dia funcione, apliquei uma técnica de preenchimento de série temporal:
* **Forward Fill:** Utilização de uma *Window Function* (`rowsBetween(Window.unboundedPreceding, 0)`) associada à função `last(ignorenulls=True)`. Isso propaga a cotação válida do último dia útil (ex: sexta-feira) para os dias vazios subsequentes (sábado e domingo).

In [0]:
# ==============================================================================
# Tabela: silver.tb_cotacao_dolar (Série Contínua)
# ==============================================================================
df_cotacao_bruta = spark.read.table("bronze.tb_cotacao_dolar")

# Converter a string de dataHora para DATE
df_cotacao_datas = df_cotacao_bruta.withColumn("data_cotacao", to_date(col("dataHoraCotacao")))

# Aplicar a mesma janela de Forward Fill utilizada na tabela financeira
window_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)

df_silver_cotacao = df_cotacao_datas.withColumn("cotacao_dolar", last("cotacaoCompra", ignorenulls=True).over(window_ffill)) \
                                    .select("data_cotacao", "cotacao_dolar") \
                                    .dropDuplicates(["data_cotacao"])

df_silver_cotacao.write.format("delta").mode("overwrite").saveAsTable("silver.tb_cotacao_dolar")
print("Tabela silver.tb_cotacao_dolar processada e gravada com sucesso!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela silver.tb_cotacao_dolar processada e gravada com sucesso!


### 6. Desnormalização de Gêneros (Split & Explode)
Os dados brutos agrupavam múltiplos gêneros em uma única string. Para adequar os dados à futura modelagem dimensional (*Star Schema*):
* **Padronização de Delimitadores:** Substituição de pontos e vírgulas (`;`) por vírgulas padrão (`,`).
* **Explosão de Arrays:** Uso de `split` seguido de `explode` para gerar uma nova linha para cada gênero associado a um filme.
* **Higienização Avançada:** Remoção de resíduos de *Column Shift* (strings numéricas isoladas ou textos excessivamente longos que não representam gêneros reais) utilizando Regex e filtros de comprimento de string.

In [0]:
from pyspark.sql.functions import col, split, explode, trim, length, initcap, regexp_replace

# Ler a tabela bruta
df_credits_bruta = spark.read.table("bronze.tb_credits_and_tags")

# ==============================================================================
# Tabela: silver.tb_generos
# ==============================================================================
# 1. Tratar inconsistência de separadores: substitui ';' e '|' por ','
df_genres_array = df_credits_bruta.withColumn(
    "genres_arr", 
    split(regexp_replace(col("genres"), "[;|]", ","), ",")
)

# 2. Explodir o array
df_genres_expl = df_genres_array.select(
    col("id").alias("id_filme"),
    explode(col("genres_arr")).alias("nome_genero")
)

# 3. DOMÍNIO OFICIAL (Whitelist baseada no TMDB)
generos_oficiais = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", 
    "Documentary", "Drama", "Family", "Fantasy", "History", 
    "Horror", "Music", "Mystery", "Romance", "Science Fiction", 
    "Sci-Fi", "Tv Movie", "Thriller", "War", "Western"
]

# 4. Limpeza Definitiva de Column Shift
df_silver_generos = df_genres_expl.withColumn("nome_genero", initcap(trim(col("nome_genero")))) \
    .filter(col("nome_genero").isin(generos_oficiais)) \
    .dropDuplicates(["id_filme", "nome_genero"])

# Gravar na Silver
df_silver_generos.write.format("delta").mode("overwrite").saveAsTable("silver.tb_generos")
print("Tabela silver.tb_generos processada com domínio oficial e gravada com sucesso!")

Tabela silver.tb_generos processada com domínio oficial e gravada com sucesso!


### 7. Consolidação de Entidades: Elenco, Direção, Roteiro e Produtoras
Criação de uma base unificada de entidades participantes por meio de programação funcional:
* **Modularização:** Criação da função Python `extrair_entidades()`, que encapsula a lógica repetitiva de *split*, *explode*, *trim* e higienização (Regex/Filtro de lixo espacial).
* **Classificação:** Adição da coluna explícita `tipo_entidade` para categorizar o papel de cada participante ('Ator', 'Diretor', 'Roteirista', 'Produtora').
* **Consolidação:** Empilhamento dos quatro DataFrames distintos utilizando `unionByName`, gerando uma tabela padronizada e limpa pronta para alimentar as dimensões de Pessoas e Empresas na camada Gold.

In [0]:
# ==============================================================================
# Tabela: silver.tb_pessoas_empresas
# ==============================================================================
# Função auxiliar para processar as 4 colunas de forma padronizada
def extrair_entidades(coluna_origem, tipo_entidade):
    return df_credits_bruta.withColumn("arr", split(regexp_replace(col(coluna_origem), ";", ","), ",")) \
                           .select(col("id").alias("id_filme"), explode(col("arr")).alias("nome_entidade")) \
                           .withColumn("nome_entidade", trim(col("nome_entidade"))) \
                           .filter(col("nome_entidade") != "") \
                           .filter(~col("nome_entidade").rlike("^[0-9]+$")) \
                           .filter(length(col("nome_entidade")) < 100) \
                           .withColumn("nome_entidade", initcap(col("nome_entidade"))) \
                           .withColumn("tipo_entidade", lit(tipo_entidade)) \
                           .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])

# Extrair e mapear os tipos exigidos
df_atores = extrair_entidades("cast", "Ator")
df_diretores = extrair_entidades("directors", "Diretor")
df_roteiristas = extrair_entidades("writers", "Roteirista")
df_produtoras = extrair_entidades("production_companies", "Produtora")

# Consolidar tudo num único DataFrame
df_silver_pessoas_empresas = df_atores.unionByName(df_diretores) \
                                      .unionByName(df_roteiristas) \
                                      .unionByName(df_produtoras)

df_silver_pessoas_empresas.write.format("delta").mode("overwrite").saveAsTable("silver.tb_pessoas_empresas")
print("Tabela silver.tb_pessoas_empresas processada e gravada com sucesso!")

Tabela silver.tb_pessoas_empresas processada e gravada com sucesso!
